# GraphSLAM Implementation - E. Mosca 925279

*This file is meant to be ran on Colab*

This notebook implements (full/offline) GraphSLAM on Scenario 1 from the KIT Velodyne-SLAM dataset.

Some details about the dataset:

- (Velodyne LiDAR) + INS (Inertial Navigation System)
- 2513 LiDAR scans stored as PNG range images (16-bit PNG) + INS configuration file with poses

The objective is therefore to:

- Build a pose graph where nodes are robot poses and edges are spatial constraints
- Use point cloud registration (e.g. ICP) to find relative transformations between consecutive scans, incorporating IMU data as initial estimates for registration.
- Perform global optimization to find the best consistent set of poses, minimizing the sum of squared errors across constraints
- Build a final map by aligning all point clouds with optimized poses

### Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import urllib.request
import zipfile
import tempfile
from scipy.spatial.transform import Rotation 
from scipy.optimize import least_squares 
import cv2  
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
import time

In [2]:
!pip install open3d

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.6 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [3]:
import open3d as o3d # For point cloud processing

### Download and Prepare Dataset

The cell below directly installs the dataset in the Colab environment

In [ ]:
# Dataset configuration
base_download_url = 'https://www.mrt.kit.edu/z/publ/download/velodyneslam/data/scenario1.zip'

data_folder = Path(os.path.join(os.getcwd(), "data")) # data folder to store dataset
zip_filename = data_folder / "scenario1.zip"
point_cloud_folder = data_folder / 'scenario1' # folder containing extracted data
num_expected_files = 2513

# Create data folder if it doesn't exist
data_folder.mkdir(parents=True, exist_ok=True)

# Download dataset if not present or incomplete
scan_files = list(point_cloud_folder.glob('scan*.png')) if point_cloud_folder.exists() else []

# Check if the dataset needs to be downloaded and extracted
if not point_cloud_folder.exists() or len(scan_files) < num_expected_files:
    print(f'Downloading scenario1.zip (153 MB) to {data_folder}...')
    urllib.request.urlretrieve(base_download_url, zip_filename)

    print('Extracting zip file...')
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
         zip_ref.extractall(data_folder)

    print('Download and extraction complete!')
else:
    print(f'Dataset already downloaded: {len(scan_files)} files found')

Extracting zip file...
Download and extraction complete!


## Helper Functions

Now, functions for processing data will be implemented in Python

### Read INS config file & Read dataset

- Load and combine point cloud filenames with INS data
- Read INS configuration file (imu.cfg) with timestamps, positions, orientations
- Create a pandas DataFrame combining point cloud files with corresponding INS readings
- Removes unnecessary columns and one bad row (index 1447), which has an impossible date (1970-01-01)


In [ ]:
def read_ins_config_file(filename):
    """
    Read INS (Inertial Navigation System) configuration file.

    The file contains semicolon-delimited data with:
    - Timestamps: Date/time of measurement
    - Position: X, Y, Z coordinates in meters
    - Orientation: Heading, Pitch, Roll in degrees
    - Velocities: V_X, V_Y, V_ZDown
    - Angular velocities: Omega_Heading, Omega_Pitch, Omega_Roll
    - GPS data: Num_Satellites, Latitude, Longitude, Altitude

    Returns: pandas DataFrame with datetime index
    """
    # Read INS data file (semicolon-delimited, starts at line 8)
    column_names = ["Timestamps", "Num_Satellites", "Latitude", "Longitude",
                   "Altitude", "Heading", "Pitch", "Roll", "Omega_Heading",
                   "Omega_Pitch", "Omega_Roll", "V_X", "V_Y", "V_ZDown",
                   "X", "Y", "Z"]

    # Use usecols to specify exactly which columns to use (handles trailing semicolons)
    ins_data = pd.read_csv(filename, sep=';', skiprows=7, names=column_names,
                          usecols=range(len(column_names)), engine='python')
    ins_data['Timestamps'] = pd.to_datetime(ins_data['Timestamps'],
                                            format='%Y-%m-%d %H:%M:%S.%f')
    ins_data.set_index('Timestamps', inplace=True)

    return ins_data

def read_dataset(data_folder, point_cloud_pattern='scan*.png'):
    """
    Read complete dataset: point cloud filenames + INS data.

    Combines:
    - List of point cloud PNG files (sorted)
    - INS measurements at corresponding timestamps

    Returns: DataFrame with columns ['PointCloudFileName', 'Heading', 'Pitch',
                                      'Roll', 'X', 'Y', 'Z', ...]
    """
    # Get sorted list of point cloud files
    point_cloud_folder = os.path.join(data_folder, 'scenario1').replace("\\", "/")
    point_cloud_files = sorted(Path(point_cloud_folder).glob(point_cloud_pattern))

    # Read INS data
    imu_config_file = os.path.join(point_cloud_folder, 'imu.cfg').replace("\\", "/")
    ins_data = read_ins_config_file(imu_config_file)

    # Delete bad row (index 1447 has corrupted data i.e. timestamp from 1970)
    ins_data = ins_data.drop(ins_data.index[1447])

    # Remove unused columns
    columns_to_remove = ['Num_Satellites', 'Latitude', 'Longitude', 'Altitude',
                         'Omega_Heading', 'Omega_Pitch', 'Omega_Roll',
                         'V_X', 'V_Y', 'V_ZDown']
    ins_data = ins_data.drop(columns=columns_to_remove)

    # Add point cloud filenames as first column
    dataset_table = ins_data.copy()
    dataset_table.insert(0, 'PointCloudFileName',
                        [str(f) for f in point_cloud_files])

    return dataset_table

# Load dataset
dataset_table = read_dataset(data_folder)
display(dataset_table.head())

,PointCloudFileName,Heading,Pitch,Roll,X,Y,Z
Timestamps,,,,,,,
2010-06-13 06:27:31.580021238,/content/data/scenario1/scan00000.png,1.915418,0.007438,-0.019888,-2889.1135,-2183.9123,116.46786
2010-06-13 06:27:31.684693525,/content/data/scenario1/scan00001.png,1.915431,0.007402,-0.019853,-2889.1132,-2183.9108,116.46773
2010-06-13 06:27:31.788636427,/content/data/scenario1/scan00002.png,1.915367,0.007382,-0.019806,-2889.1126,-2183.9082,116.46799
2010-06-13 06:27:31.892944145,/content/data/scenario1/scan00003.png,1.915377,0.007459,-0.019828,-2889.1129,-2183.9054,116.46943
2010-06-13 06:27:31.997256227,/content/data/scenario1/scan00004.png,1.915367,0.007374,-0.019843,-2889.1134,-2183.9035,116.46964


### INS dataset column descriptions

- Timestamps – date/time of each measurement; used to align IMU/INS readings with each LiDAR scan.
- Heading – yaw angle (degrees, rotation about Z); the primary orientation term used to initialise ICP and form relative transforms.
- Pitch / Roll – tilt angles about Y and X; usually small for a ground vehicle.
- X, Y, Z – position in the local INS Cartesian frame; directly used to compute relative translations between successive frames.

### Read Point Cloud from PNG file 

- Convert PNG range image to 3D point cloud
- PNG files store 16-bit distance values (divide by 500 to get meters)
- Each pixel represents a LiDAR measurement
- Uses spherical coordinates (yaw, pitch, range) --> change to Cartesian (x, y, z)

Conversions:
- Yaw angles: Column index maps linearly from 180° to -180°
- Pitch angles: Fixed per row (64 laser beams at different vertical angles)
- Range = pixel_value / 500 meters

Coordinate system: Transform to vehicle frame (X=forward, Y=left, Z=up)

In [ ]:
def read_point_cloud_from_file(filename):
    """
    Read point cloud from PNG range image file.

    Dataset format (from DATAFORMAT.txt):
    - 16-bit PNG where pixel value / 500 = distance in meters
    - Image dimensions: 64 rows (laser beams) by 870 columns (azimuth angles)
    - 360° scan stored left-to-right, with left/right edges at vehicle rear
    - Zero values = invalid measurements (NaN)
    - Ignore leftmost and rightmost 10 columns due to motion artifacts

    Spherical to Cartesian conversion:
    -> x = range * cos(pitch) * sin(yaw)
    -> y = range * cos(pitch) * cos(yaw)
    -> z = -range * sin(pitch)

    Returns: Nx3 numpy array or open3d.PointCloud object
    """
    # Read 16-bit PNG and convert to range
    img = cv2.imread(filename, cv2.IMREAD_UNCHANGED)  # Read as 16-bit
    range_data = np.array(img).astype(np.float32) / 500.0
    range_data[range_data == 0] = np.nan  # Invalid measurements

    # Yaw angles: linear mapping [0..869] → [180° to -180°]
    yaw_angles = np.linspace(180, -180, 870)
    yaw_angles_rad = np.deg2rad(yaw_angles)

    # Pitch angles for 64 laser beams (from img.cfg)
    pitch_angles_deg = np.array([
        -1.9367, -1.57397, -1.30476, -0.871566, -0.57881, -0.180617,
        0.088762, 0.451829, 0.80315, 1.20124, 1.49388, 1.83324, 2.20757,
        2.54663, 2.87384, 3.23588, 3.53933, 3.93585, 4.21552, 4.5881, 4.91379,
        5.25078, 5.6106, 5.9584, 6.32889, 6.67575, 6.99904, 7.28731, 7.67877,
        8.05803, 8.31047, 8.71141, 9.02602, 9.57351, 10.0625, 10.4707, 10.9569,
        11.599, 12.115, 12.5621, 13.041, 13.4848, 14.0483, 14.5981, 15.1887,
        15.6567, 16.1766, 16.554, 17.1868, 17.7304, 18.3234, 18.7971, 19.3202,
        19.7364, 20.2226, 20.7877, 21.3181, 21.9355, 22.4376, 22.8566, 23.3224,
        23.971, 24.5066, 24.9992
    ])
    # To radians since functions expect radians
    pitch_angles_rad = np.deg2rad(pitch_angles_deg)

    # Create meshgrid for all combinations to compute Cartesian coordinates
    yaw_grid, pitch_grid = np.meshgrid(yaw_angles_rad, pitch_angles_rad)

    # Spherical to Cartesian conversion
    x = range_data * np.cos(pitch_grid) * np.sin(yaw_grid)
    y = range_data * np.cos(pitch_grid) * np.cos(yaw_grid)
    z = -range_data * np.sin(pitch_grid)

    # Transform to vehicle frame: flip X direction
    x = -x

    # Reshape to Nx3 array and remove NaN points
    points = np.stack([x.flatten(), y.flatten(), z.flatten()], axis=1)
    valid_mask = ~np.isnan(points).any(axis=1)
    points = points[valid_mask]

    # Create Open3D point cloud
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    return pcd


# Example: Read first point cloud
ptcloud = read_point_cloud_from_file(dataset_table['PointCloudFileName'].iloc[0])
print(ptcloud, ptcloud.points, "\n", np.asarray(ptcloud.points))

PointCloud with 49625 points. std::vector<Eigen::Vector3d> with 49625 elements.
Use numpy.asarray() to access data. 
 [[-2.99891565e-15 -2.44880046e+01  8.28053821e-01]
 [-1.79570320e-01 -2.48351562e+01  8.39814590e-01]
 [-3.75433069e-01 -2.59604468e+01  8.77935760e-01]
 ...
 [ 2.30601051e+00 -3.53181288e+00 -1.96680641e+00]
 [ 2.28139431e+00 -3.54991884e+00 -1.96765177e+00]
 [ 2.25372975e+00 -3.56325722e+00 -1.96596126e+00]]


### Process Point Cloud

Preprocessing point clouds before registration by:

- Removing ground plane: Using plane fitting (RANSAC)

Given:
- Ground is static and dominant --> can bias registration
- Vehicle points don't represent the environment
- Processing reduces point count --> faster ICP convergence
- Improves registration accuracy by focusing on environmental features

So try to fit plane with most inliers to the ground with RANSAC, and only keep non-ground points.

In [ ]:
def process_point_cloud(ptcloud_in):
    """
    Process point cloud by removing ground plane.

        Fit plane with RANSAC (reference normal = [0,0,1])

    Args:
        ptcloud_in: Input point cloud (open3d.PointCloud or Nx3 array)

    Returns:
        Processed point cloud with ground and ego removed
    """
    # RANSAC plane fitting for ground removal
    plane_model, inliers = ptcloud_in.segment_plane(
        distance_threshold=0.4,
        ransac_n=3,
        num_iterations=1000
    )
    # Verify plane normal points upward (positive Z)
    if plane_model[2] < 0:  # normal is [A, B, C] in Ax + By + Cz + D = 0
        plane_model = -plane_model

    # Keep non-ground points
    outlier_cloud = ptcloud_in.select_by_index(inliers, invert=True)

    return outlier_cloud

print("Point cloud processing function defined")

Point cloud processing function defined


### Compute Initial Estimate from INS

Estimate transformation between frames using IMU/INS data

- Point cloud registration (ICP) needs a good initial guess to converge correctly
- INS provides position (X, Y, Z) and orientation (heading, pitch, roll)
- Compute relative transformation from frame A to frame B using INS readings

Process:
1. Get INS pose at time t₁ and t₂
2. Convert positions: INS frame → LiDAR frame (offset: [0, -0.79, -1.73])
3. Build 4×4 transformation matrices for both poses
4. Compute relative transform: T_rel = T₁⁻¹ × T₂
5. Simplification: Ignore pitch/roll changes (assuming ground vehicle motion)

In [ ]:
def compute_initial_estimate_from_ins(ins_data_slice):
    """
    Compute initial transformation estimate from INS readings.

    INS coordinate system:
    - X points forward, Y points left, Z points up
    - Heading: rotation around Z axis

    Args:
        ins_data_slice: DataFrame slice with INS data between two frames
                       Contains columns: ['X', 'Y', 'Z', 'Heading', 'Pitch', 'Roll']

    Returns:
        4x4 transformation matrix (numpy array)
    """
    if ins_data_slice is None or len(ins_data_slice) == 0:
        # Return identity if no INS data
        return np.eye(4)

    # INS to LiDAR offset (from DATAFORMAT.txt)
    ins_to_lidar_offset = np.array([0, -0.79, -1.73])

    # Get first and last poses in the slice
    first_pose = ins_data_slice.iloc[0]
    last_pose = ins_data_slice.iloc[-1]

    # Extract positions (INS frame: X forward, Y left, Z up) and convert to LiDAR frame: swap and apply offset
    T_before = np.array([-first_pose['Y'], first_pose['X'], first_pose['Z']]) + ins_to_lidar_offset
    T_now = np.array([-last_pose['Y'], last_pose['X'], last_pose['Z']]) + ins_to_lidar_offset

    # Extract heading (yaw) only - ignore pitch/roll for ground vehicles
    heading_before = np.deg2rad(first_pose['Heading'])
    heading_now = np.deg2rad(last_pose['Heading'])

    # Build rotation matrices (Z-axis rotation)
    R_before = Rotation.from_euler('z', heading_before).as_matrix()
    R_now = Rotation.from_euler('z', heading_now).as_matrix()

    # Build 4x4 transformation matrices [R | t; 0 0 0 1]
    T_before_mat = np.eye(4)
    T_before_mat[:3, :3] = R_before
    T_before_mat[:3, 3] = T_before

    T_now_mat = np.eye(4)
    T_now_mat[:3, :3] = R_now
    T_now_mat[:3, 3] = T_now

    # Compute relative transformation: T_rel = T_before^-1 * T_now
    T_rel = np.linalg.inv(T_before_mat) @ T_now_mat

    return T_rel


print("INS transformation function defined")

INS transformation function defined


## Visualize Point Cloud Stream 

Quick visualization to verify data loading and understand the environment

This shows the raw LiDAR scans in sequence, the cell below takes a bit to run, but the output has been supplied in the "pointcloud_stream.mp4" file, so I suggest to simply watch that.


In [ ]:
import matplotlib
matplotlib.use('Agg')  # headless backend
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Visualization parameters
x_limits = [-45, 45]
y_limits = [-45, 45]
z_limits = [-10, 20]

def pc_to_image(ptcloud, x_limits=[-45,45], y_limits=[-45,45], z_limits=[-10,20], figsize=(8,6), elev=30, azim=-60):
    """Render a point cloud to a matplotlib figure and return as image array"""
    pts = np.asarray(ptcloud.points)
    if pts.size == 0:
        h = int(figsize[1]*100)
        w = int(figsize[0]*100)
        return np.zeros((h, w, 3), dtype=np.uint8)

    # Downsample for faster plotting
    max_pts = 20000
    if pts.shape[0] > max_pts:
        idx = np.random.choice(pts.shape[0], max_pts, replace=False)
        pts = pts[idx]

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], c=pts[:,2], cmap='viridis', s=0.5, linewidth=0)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)
    ax.set_zlim(z_limits)
    ax.axis('off')
    fig.tight_layout(pad=0)
    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    # Use buffer_rgba for better compatibility with newer matplotlib versions
    img_rgba = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape((h, w, 4))
    img = img_rgba[:, :, :3]  # Drop alpha channel
    plt.close(fig)
    return img

# Video parameters
output_mp4 = 'pointcloud_stream.mp4'
fps = 10
skip_frames = 2

# Prepare writer using first frame to get size
first_filename = dataset_table['PointCloudFileName'].iloc[0]
first_pc = read_point_cloud_from_file(first_filename)
first_pc = process_point_cloud(first_pc)
first_img = pc_to_image(first_pc, x_limits, y_limits, z_limits)
h, w = first_img.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_mp4, fourcc, fps, (w, h))

# Iterate through dataset, render frames and write to mp4
num_frames = len(dataset_table)
for i in range(0, num_frames, skip_frames):
    filename = dataset_table['PointCloudFileName'].iloc[i]
    pc = read_point_cloud_from_file(filename)
    pc = process_point_cloud(pc)

    # Optional: downsample to keep frames lightweight
    try:
        pc = pc.voxel_down_sample(voxel_size=0.2)
    except Exception:
        pass

    img = pc_to_image(pc, x_limits, y_limits, z_limits)
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    writer.write(img_bgr)

    if i % 200 == 0:
        print(f'Rendered frame {i}/{num_frames}')

writer.release()
print(f"Saved point cloud stream to {output_mp4}")

Rendered frame 0/2513
Rendered frame 200/2513
Rendered frame 400/2513
Rendered frame 600/2513
Rendered frame 800/2513
Rendered frame 1000/2513
Rendered frame 1200/2513
Rendered frame 1400/2513
Rendered frame 1600/2513
Rendered frame 1800/2513
Rendered frame 2000/2513
Rendered frame 2200/2513
Rendered frame 2400/2513
Saved point cloud stream to pointcloud_stream.mp4


## Building the Pose Graph 

Data Structures

Pose Graph, python equivalent of MATLAB's `pcviewset` with:
- Nodes (Views) composed of:
  - `view_id`: Unique identifier
  - `pose`: 4×4 absolute transformation matrix (world frame)
  - `point_cloud`: Associated LiDAR scan
  
- Edges (Connections): composed of:
  - `from_id`, `to_id`: Connected view IDs
  - `relative_transform`: 4×4 transformation from view A to view B
  - `information_matrix`: Uncertainty/confidence (6×6 matrix), set to identity

Algorithm Flow

1. Start with identity pose for first frame
2. For each subsequent frame:
   - Load and preprocess point cloud (removing ground plane)
   - Compute INS-based initial estimate (obtaining T_rel matrix)
   - Register with previous frame using ICP
   - Add node with absolute pose
   - Add edge with relative transformation  
3. Optimization:
   - Convert pose graph to optimization problem
   - Minimize error across all edges
   - Update all poses to be globally consistent

In [ ]:
# Initialize random seed for reproducibility
np.random.seed(0)

# Data structure: Pose Graph
# In Python, we can use a class or dictionaries to represent the graph
class PoseGraphNode:
    """Represents a node in the pose graph"""
    def __init__(self, view_id, pose, point_cloud=None):
        self.view_id = view_id
        self.pose = pose  # 4x4 transformation matrix
        self.point_cloud = point_cloud

class PoseGraphEdge:
    """Represents an edge (constraint) in the pose graph"""
    def __init__(self, from_id, to_id, relative_transform, information=None):
        self.from_id = from_id
        self.to_id = to_id
        self.relative_transform = relative_transform  # 4x4 matrix
        self.information = information if information is not None else np.eye(6)

class PoseGraph:
    """pose graph structure"""
    def __init__(self):
        self.nodes = {}  # view_id -> PoseGraphNode
        self.edges = []  # List of PoseGraphEdge

    def add_node(self, view_id, pose, point_cloud=None):
        """Add a view/node to the graph"""
        self.nodes[view_id] = PoseGraphNode(view_id, pose, point_cloud)

    def add_edge(self, from_id, to_id, relative_transform, information=None):
        """Add a connection/edge between two views"""
        self.edges.append(PoseGraphEdge(from_id, to_id, relative_transform, information))

    def get_pose(self, view_id):
        """Get absolute pose for a view"""
        return self.nodes[view_id].pose

    def update_poses(self, optimized_poses):
        """Update poses after optimization"""
        for view_id, pose in optimized_poses.items():
            if view_id in self.nodes:
                self.nodes[view_id].pose = pose

# Initialize pose graph
pose_graph = PoseGraph()

# Initialize transformations
abs_transform = np.eye(4)  # Current absolute pose
rel_transform = np.eye(4)  # Relative transformation between frames

print("Pose graph data structures initialized")

Pose graph data structures initialized


### Sequential Registration

Process each frame:
1. Load and preprocess point cloud
2. Get INS-based initial estimate  
3. Perform ICP registration with previous frame
4. Accumulate pose and add to graph

In [ ]:
# Build pose graph through sequential registration
skip_frames = 2
num_frames = len(dataset_table)

# Process first frame
view_id = 0
filename = dataset_table['PointCloudFileName'].iloc[0]
prev_ptcloud = read_point_cloud_from_file(filename)
prev_ptcloud = process_point_cloud(prev_ptcloud)

# Add first node with identity pose
pose_graph.add_node(view_id, abs_transform.copy(), prev_ptcloud)

# Process subsequent frames
for n in range(skip_frames, num_frames, skip_frames):
     view_id += 1

     # Load and process current point cloud
     filename = dataset_table['PointCloudFileName'].iloc[n]
     curr_ptcloud = read_point_cloud_from_file(filename)
     curr_ptcloud = process_point_cloud(curr_ptcloud)

     ## Optional: Downsample for faster ICP
     curr_ptcloud = curr_ptcloud.voxel_down_sample(voxel_size=0.5)
     prev_ptcloud_downsampled = prev_ptcloud.voxel_down_sample(voxel_size=0.5)

     # Compute initial estimate from INS data
     ins_slice = dataset_table.iloc[n-skip_frames:n+1]
     init_transform = compute_initial_estimate_from_ins(ins_slice)

     # Perform ICP registration
     threshold = 2.0  # Maximum correspondence distance
     reg_result = o3d.pipelines.registration.registration_icp(
          curr_ptcloud, prev_ptcloud_downsampled, threshold, init_transform,
          o3d.pipelines.registration.TransformationEstimationPointToPoint(),
          o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=50)
      )
     rel_transform = reg_result.transformation

     # Update absolute pose: T_abs_new = T_abs_prev * T_rel
     abs_transform = abs_transform @ rel_transform

     # Add node and edge to graph
     pose_graph.add_node(view_id, abs_transform.copy(), curr_ptcloud)
     pose_graph.add_edge(view_id - 1, view_id, rel_transform)

     # Update for next iteration
     prev_ptcloud = curr_ptcloud

     # Progress indicator
     if view_id % 100 == 0:
          print(f"Processed frame {view_id}/{num_frames//skip_frames}")

print(f"\nSequential registration complete!")
print(f"Built pose graph with:")
print(f"  - {len(pose_graph.nodes)} nodes")
print(f"  - {len(pose_graph.edges)} edges (all sequential)")


Processed frame 100/1256
Processed frame 200/1256
Processed frame 300/1256
Processed frame 400/1256
Processed frame 500/1256
Processed frame 600/1256
Processed frame 700/1256
Processed frame 800/1256
Processed frame 900/1256
Processed frame 1000/1256
Processed frame 1100/1256
Processed frame 1200/1256

Sequential registration complete!
Built pose graph with:
  - 1257 nodes
  - 1256 edges (all sequential)


## Pose Graph Optimization

Finding globally consistent poses that minimize error across all edges

- Variables: All robot poses (x, y, z, roll, pitch, yaw) for each frame
- Objective: Minimize sum of squared errors across all edges, errors defined as distance between predicted and measured relative transform.
- Least squares minimization with Levenberg-Marquardt method: Iterative optimization balancing gradient descent and Newton's method

DIsclaimer: on Colab this takes more than an hour, you can load the 'optimized_poses' pickle file in the cell after this one directly.

In [ ]:
def optimize_pose_graph_scipy(pose_graph):
    """
    Optimize pose graph using scipy.optimize.least_squares (Levenberg-Marquardt method).

    Encodes each pose as 6D vector: [x, y, z, roll, pitch, yaw]
    Minimizes the sum of squared edge constraint errors using Levenberg-Marquardt.

    Args:
        pose_graph: PoseGraph object with nodes and edges

    Returns:
        Dictionary of optimized poses {view_id: 4x4 matrix}
    """
    node_ids = sorted(pose_graph.nodes.keys())
    n_nodes = len(node_ids)

    def poses_to_vector(poses_dict):
        """Convert pose dict to flat vector"""
        vector = []
        for nid in node_ids:
            T = poses_dict[nid]
            t = T[:3, 3]  # Translation
            R = T[:3, :3]  # Rotation matrix
            euler = Rotation.from_matrix(R).as_euler('zyx')  
            vector.extend([*t, *euler])
        return np.array(vector)

    def vector_to_poses(vector):
        """Convert flat vector to pose dict"""
        poses = {}
        for i, nid in enumerate(node_ids):
            offset = i * 6
            t = vector[offset:offset+3]
            euler = vector[offset+3:offset+6]
            R = Rotation.from_euler('zyx', euler).as_matrix()
            T = np.eye(4)
            T[:3, :3] = R
            T[:3, 3] = t
            poses[nid] = T
        return poses

    def residual_function(vector):
        """Compute residual vector for least_squares (one element per constraint)"""
        poses = vector_to_poses(vector)
        residuals = []

        # Penalty: first pose should be identity
        first_id = node_ids[0]
        T0 = poses[first_id]
        residuals.extend(T0[:3, 3] * 1e2)  # Large weight on translation
        residuals.extend((T0[:3, :3] - np.eye(3)).flatten() * 1e2)  # Large weight on rotation

        # Edge constraint residuals
        for edge in pose_graph.edges:
            T_from = poses[edge.from_id]
            T_to = poses[edge.to_id]

            # Predicted vs measured relative transform
            T_rel_pred = np.linalg.inv(T_from) @ T_to
            T_rel_meas = edge.relative_transform

            # Translation residual
            dt = T_rel_pred[:3, 3] - T_rel_meas[:3, 3]

            # Rotation residual as a 3-vector (angle-axis) so each edge
            # contributes 3 rotation residuals instead of a single scalar.
            # This increases the total number of residuals so Levenberg-
            # Marquardt ('lm') can be used (requires m >= n).
            dR = T_rel_pred[:3, :3] @ T_rel_meas[:3, :3].T
            rot_vec = Rotation.from_matrix(dR).as_rotvec()  # 3-vector

            # Apply information weighting... information matrices are identity in 6x6
            weight = 1.0
            if edge.information is not None:
                weight = np.sqrt(np.linalg.det(edge.information))

            residuals.extend(dt * weight)
            residuals.extend(rot_vec * weight)

        return np.array(residuals)

    # Initial parameter vector from current poses
    initial_vector = poses_to_vector({nid: pose_graph.nodes[nid].pose for nid in node_ids})

    print("Starting pose graph optimization (Levenberg-Marquardt)...")
    print(f"  Nodes: {len(pose_graph.nodes)}")
    print(f"  Edges: {len(pose_graph.edges)}")

    # Optimize using Levenberg-Marquardt
    result = least_squares(residual_function, initial_vector,
                          method='lm', max_nfev=1000, verbose=1)

    optimized_vector = result.x
    optimized_poses = vector_to_poses(optimized_vector)

    print(f"Optimization complete (Levenberg-Marquardt)")
    print(f"  Final cost: {result.cost:.6f}")
    print(f"  Iterations: {result.nfev}")
    print(f"  Success: {result.success}")

    return optimized_poses


# Run optimization# Update pose graph with optimized poses

optimized_poses = optimize_pose_graph_scipy(pose_graph)
pose_graph.update_poses(optimized_poses)

Starting pose graph optimization (Levenberg-Marquardt)...
  Nodes: 1257
  Edges: 1256
`xtol` termination condition is satisfied.
Function evaluations 2, initial cost 1.5504e-21, final cost 3.9393e-25, first-order optimality 1.28e-13.
Optimization complete (Levenberg-Marquardt)
  Final cost: 0.000000
  Iterations: 2
  Success: True


In [ ]:
# persist optimized poses so we don't have to rerun the optimizer
#import pickle
#with open('optimized_poses.pkl','wb') as f:
#    pickle.dump(optimized_poses, f)
#print('Saved optimized poses to optimized_poses.pkl')

Saved optimized poses to optimized_poses.pkl


Upload optimized_poses.pkl on the Colab environment

In [10]:
#optional optimized poses load
import pickle
with open('optimized_poses.pkl','rb') as f:
    optimized_poses = pickle.load(f)

## Build Final Map

Merging all point clouds using optimized poses

- Transform each point cloud to world frame using its optimized pose
- Merge all transformed clouds
- Downsample to uniform grid (voxel grid filter)

In [ ]:
def build_map(pose_graph, voxel_size=0.2):
    """
    Build final map by merging all point clouds with optimized poses.

    Args:
        pose_graph: PoseGraph with optimized poses
        voxel_size: Grid size for downsampling (meters)

    Returns:
        Combined point cloud (open3d.PointCloud)
    """
    combined_cloud = o3d.geometry.PointCloud()

    for view_id, node in pose_graph.nodes.items():
        # Transform point cloud to world frame
        ptcloud = node.point_cloud.transform(node.pose)

        # Add to combined cloud
        combined_cloud += ptcloud

    # Downsample to uniform grid (reduces redundancy)
    map_cloud = combined_cloud.voxel_down_sample(voxel_size=voxel_size)

    print(f"Map created: {len(map_cloud.points)} points")
    return map_cloud

# Build final map
map_grid_size = 0.2  # 20cm voxel grid
final_map = build_map(pose_graph, voxel_size=map_grid_size)

print("Map building function defined")

Map created: 3472123 points
Map building function defined


## Visualizing the Result

- Final Bird's-Eye-view map in 2D with altitude color coding
- Trajectory: Optimized robot poses connected as a path

The result is saved as PNG file to disk (which should already be available), but can re-run anyways

In [31]:
def visualize_map_and_trajectory(final_map, pose_graph, output_path='map_trajectory.png'):
    """
    Visualize the final map with robot trajectory and save to PNG (no window).

    Args:
        final_map: Combined point cloud
        pose_graph: PoseGraph with optimized poses
        output_path: Path to save the output PNG file
    """
    # Extract trajectory points
    trajectory_points = []
    for view_id in sorted(pose_graph.nodes.keys()):
        pose = pose_graph.nodes[view_id].pose
        position = pose[:3, 3]
        trajectory_points.append(position)
    trajectory_points = np.array(trajectory_points)

    # Extract point cloud data
    points = np.asarray(final_map.points)

    # Subsample if large
    max_points = 100_000
    if len(points) > max_points:
        idx = np.random.choice(len(points), max_points, replace=False)
        plot_points = points[idx]
    else:
        plot_points = points

    # Use Z for color (height-based gradient)
    z_vals = plot_points[:, 2]

    if len(z_vals) == 0:
        print("Warning: Point cloud is empty, skipping visualization.")
        return

    z_min, z_max = z_vals.min(), z_vals.max()
    z_norm = (z_vals - z_min) / (z_max - z_min + 1e-9)

    # --- Plot: top-down (X-Y) view ---
    fig, ax = plt.subplots(figsize=(16, 9), facecolor='white')

    scatter = ax.scatter(
        plot_points[:, 0],
        plot_points[:, 1],
        c=z_norm,
        cmap='jet',
        s=0.5,
        alpha=0.8,
        linewidths=0
    )

    # Draw trajectory
    if len(trajectory_points) > 1:
        ax.plot(
            trajectory_points[:, 0],
            trajectory_points[:, 1],
            color='red',
            linewidth=1,
            alpha=0.7,
            label='Trajectory',
            zorder=5
        )
    ax.scatter(
        trajectory_points[:, 0],
        trajectory_points[:, 1],
        color='red',
        s=15,
        zorder=6
    )

    # Colorbar
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.02, pad=0.01)
    cbar.set_label('Height (Z)', fontsize=10)
    cbar.set_ticks([0, 0.5, 1])
    cbar.set_ticklabels(['Low', 'Mid', 'High'])

    ax.set_title('GraphSLAM Result: Map + Trajectory (Top-Down View)', fontsize=14)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_aspect('equal')
    ax.legend(loc='upper right')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"Map saved to: {output_path}")

# Save result
visualize_map_and_trajectory(final_map, pose_graph, output_path='map_trajectory.png')

Map saved to: map_trajectory.png


## Commenting the Results

The final map and the optimized trajectory are very much in line with the world observed in the point cloud trajectory video, as well as the ground truth available [here](https://www.mrt.kit.edu/z/publ/download/velodyneslam/dataset.html); the trajectory alone can be inferred to be correct from watching the video. When comparing with the ground truth, the map has some imperfections:

- The merged point clouds give rise to non-existent roads, as those visible in red (high altitude) in the bottom half of the map_trajectory image, but also in other places there are some "ghost roads" which might not be present in the real world.
- The altitude in the map isn't exactly recovered, as can be seen from looking at the ground truth image from scenario 1, a much larger portion of the map should have high altitude (looks like right side of the map should be much more red, assuming same altitude colormapping).


Thanks for your time